# 13장. LLM을 평가하는 방법

같은 모델도 채점 방법에 따라 점수가 달라진다. 9장의 세 모델로 한국어 객관식 벤치마크(KoBEST)를 두 방식으로 채점하고, LLM에게 판정을 맡길 때의 순서·길이 편향을 재고, 벤치마크 문항을 미리 학습시켜 오염된 점수를 만들어 본다.

**준비:** `pip install "transformers<5" huggingface_hub`. 9장에서 받은 Qwen3-0.6B, HyperCLOVA X SEED 0.5B, Kanana 1.5 2.1B와, KoBEST의 `copa/test.jsonl`, `hellaswag/test.jsonl`(Hugging Face `skt/kobest_v1`, CC BY-SA 4.0)이 `data/`에 있어야 한다. 저자의 맥(CPU 스레드 2개)에서 끝까지 실행하는 데 46분쯤 걸렸다.

## 13.1 객관식 벤치마크를 로그 확률로 채점하기

In [ ]:
import os, re, json, math, time, sys, random, platform, warnings, collections
import numpy as np
import torch
import torch.nn.functional as F
os.environ["HF_HUB_DISABLE_XET"] = "1"
warnings.filterwarnings("ignore")
torch.set_num_threads(2)
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

PATH = {"Qwen3-0.6B": "data/models/Qwen__Qwen3-0.6B",
        "HyperCLOVAX-SEED-0.5B": "data/models/naver-hyperclovax__HyperCLOVAX-SEED-Text-Instruct-0.5B",
        "Kanana-1.5-2.1B": "data/models/kakaocorp__kanana-1.5-2.1b-instruct-2505"}
STOP = {"Qwen3-0.6B": ["<|im_end|>", "<|endoftext|>"], "HyperCLOVAX-SEED-0.5B": ["<|endofturn|>", "<|stop|>"],
        "Kanana-1.5-2.1B": ["<|eot_id|>"]}                  # 9장에서 확인한 끝 토큰
MODELS = list(PATH)
tok = {n: AutoTokenizer.from_pretrained(PATH[n]) for n in MODELS}
load = lambda n: AutoModelForCausalLM.from_pretrained(PATH[n], dtype=torch.float32).eval()
def chat(n, text):
    """9장처럼 그 모델의 채팅 템플릿에 질문을 넣은 토큰(Qwen3는 생각 모드를 끈다)."""
    extra = {"enable_thinking": False} if n.startswith("Qwen3") else {}
    return tok[n].apply_chat_template([{"role": "user", "content": text}], tokenize=True, add_generation_prompt=True, **extra)
print(sys.version.split()[0], torch.__version__, transformers.__version__, platform.machine(), torch.get_num_threads())

In [ ]:
COPA = [json.loads(l) for l in open("data/kobest/copa/test.jsonl", encoding="utf-8")]
HS = [json.loads(l) for l in open("data/kobest/hellaswag/test.jsonl", encoding="utf-8")]
print(f"COPA {len(COPA)}문항(선택지 2개), HellaSwag {len(HS)}문항(선택지 4개)")
print("COPA 정답 분포", collections.Counter(c["label"] for c in COPA), " HellaSwag 정답 분포", collections.Counter(h["label"] for h in HS))
c = COPA[1]
print(f"COPA 예: {c['premise']} ({c['question']}) ① {c['alternative_1']} ② {c['alternative_2']} → 정답 {c['label'] + 1}")

In [ ]:
CONNECT = {"원인": " 왜냐하면", "결과": " 그래서"}           # COPA는 '원인'이면 '왜냐하면', '결과'면 '그래서'로 잇는다
def items(task):
    """(문맥, [선택지들], 정답 번호). 선택지 앞에는 빈칸 하나를 둔다."""
    if task == "COPA":
        return [(c["premise"].rstrip(".") + CONNECT[c["question"]], [" " + c["alternative_1"], " " + c["alternative_2"]], c["label"]) for c in COPA]
    return [(h["context"], [" " + h[f"ending_{i}"] for i in range(1, 5)], h["label"]) for h in HS]

def cont_logp(model, t, pairs, batch=16):
    """(문맥, 이어질 글) 묶음마다 이어질 글 토큰의 log 확률 합과 토큰 수. 채팅 템플릿 없이 글을 그대로 잇는다."""
    out = []
    for k in range(0, len(pairs), batch):
        rows = []
        for ctx, cont in pairs[k:k + batch]:
            a, b = t(ctx)["input_ids"], t(cont, add_special_tokens=False)["input_ids"]
            rows.append((a + b, len(a), len(b)))
        L = max(len(r[0]) for r in rows); pad = t.pad_token_id if t.pad_token_id is not None else 0
        ids = torch.tensor([r[0] + [pad] * (L - len(r[0])) for r in rows])
        mask = torch.tensor([[1] * len(r[0]) + [0] * (L - len(r[0])) for r in rows])
        with torch.inference_mode():
            lp = F.log_softmax(model(input_ids=ids, attention_mask=mask).logits[:, :-1], -1)
        got = torch.gather(lp, 2, ids[:, 1:, None])[..., 0]
        for i, (_, a, b) in enumerate(rows):
            out.append((got[i, a - 1:a - 1 + b].sum().item(), b))
    return out

def score_ll(model, t, task):
    """선택지마다 log 확률을 구해, 합·토큰당 평균·글자당 평균 세 가지로 고른 정답률."""
    its = items(task)
    flat = [(ctx, o) for ctx, opts, _ in its for o in opts]
    lp = cont_logp(model, t, flat)
    k, right = 0, collections.Counter()
    for ctx, opts, y in its:
        s = lp[k:k + len(opts)]; k += len(opts)
        for name, f in [("합", lambda v, o: v[0]), ("토큰당", lambda v, o: v[0] / v[1]), ("글자당", lambda v, o: v[0] / len(o.strip()))]:
            right[name] += int(np.argmax([f(v, o) for v, o in zip(s, opts)]) == y)
    return {name: right[name] / len(its) for name in ["합", "토큰당", "글자당"]}

LL, took = {}, {}
for n in MODELS:
    model = load(n); lap = time.perf_counter()
    for task in ["COPA", "HellaSwag"]:
        LL[n, task] = score_ll(model, tok[n], task)
        print(f"{n:<22} {task:<9}", "  ".join(f"{k} {v:.1%}" for k, v in LL[n, task].items()), flush=True)
    took[n] = time.perf_counter() - lap
    del model
print("무작위로 고르면 COPA 50%, HellaSwag 25%")

## 13.2 답을 쓰게 해서 채점하기

In [ ]:
ASK = "다음 문장의 {} 가장 알맞은 것을 고르세요. 번호 하나로만 답하세요.\n문장: {}\n1. {}\n2. {}"
JOSA = {"원인": "원인으로", "결과": "결과로"}
def mc_prompt(c, swap=False):
    a, b = (c["alternative_2"], c["alternative_1"]) if swap else (c["alternative_1"], c["alternative_2"])
    return ASK.format(JOSA[c["question"]], c["premise"], a, b)

def answer(model, n, text, max_new=6):
    x = torch.tensor([chat(n, text)]); t = tok[n]
    stops = [t.convert_tokens_to_ids(s) for s in STOP[n]]
    with torch.inference_mode():
        g = model.generate(x, attention_mask=torch.ones_like(x), max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None, top_k=None,
                           eos_token_id=stops, pad_token_id=t.pad_token_id if t.pad_token_id is not None else stops[0])
    return t.decode(g[0, x.shape[1]:], skip_special_tokens=True).strip()

SUB = COPA[:300]
GEN, SAMPLE = {}, {}
for n in MODELS:
    model = load(n)
    res = {}
    for swap in [False, True]:
        outs = [answer(model, n, mc_prompt(c, swap)) for c in SUB]
        num = [(re.search(r"[12]", o).group(0) if re.search(r"[12]", o) else None) for o in outs]
        truth = [str((c["label"] if not swap else 1 - c["label"]) + 1) for c in SUB]
        res[swap] = {"acc": float(np.mean([a == b for a, b in zip(num, truth)])), "bad": float(np.mean([a is None for a in num])),
                     "one": float(np.mean([a == "1" for a in num])), "pick": [None if a is None else (int(a) - 1 if not swap else 2 - int(a)) for a in num]}
        SAMPLE[n, swap] = outs[:3]
    same = np.mean([p is not None and p == q for p, q in zip(res[False]["pick"], res[True]["pick"])])
    GEN[n] = {"acc": res[False]["acc"], "acc_swap": res[True]["acc"], "bad": (res[False]["bad"] + res[True]["bad"]) / 2,
              "one": (res[False]["one"] + res[True]["one"]) / 2, "same": float(same)}
    print(f"{n:<22} 정답률 {GEN[n]['acc']:.1%}, 순서를 바꾸면 {GEN[n]['acc_swap']:.1%}, 번호를 못 읽은 답 {GEN[n]['bad']:.1%}, "
          f"'1'이라고 답한 비율 {GEN[n]['one']:.1%}, 두 순서에서 같은 선택지를 고른 문항 {GEN[n]['same']:.1%}", flush=True)
    print("   답 예:", SAMPLE[n, False])
    del model
print("정답이 1번인 문항", f"{np.mean([c['label'] == 0 for c in SUB]):.1%}")

## 13.3 LLM에게 판정을 맡기기

In [ ]:
QA = [("대한민국의 수도는 어디인가요?", "서울입니다.", "부산입니다."), ("일본의 수도는 어디인가요?", "도쿄입니다.", "오사카입니다."),
      ("물의 화학식은 무엇인가요?", "H2O입니다.", "CO2입니다."), ("1년은 몇 개월인가요?", "12개월입니다.", "10개월입니다."),
      ("세종대왕이 만든 문자는 무엇인가요?", "한글입니다.", "한자입니다."), ("태양계에서 가장 큰 행성은 무엇인가요?", "목성입니다.", "토성입니다."),
      ("지구에서 가장 넓은 바다는 무엇인가요?", "태평양입니다.", "대서양입니다."), ("7 곱하기 8은 얼마인가요?", "56입니다.", "54입니다."),
      ("100에서 37을 빼면 얼마인가요?", "63입니다.", "73입니다."), ("남한에서 가장 높은 산은 무엇인가요?", "한라산입니다.", "지리산입니다."),
      ("임진왜란이 일어난 해는 언제인가요?", "1592년입니다.", "1492년입니다."), ("삼각형 세 내각의 합은 몇 도인가요?", "180도입니다.", "360도입니다."),
      ("사과를 영어로 하면 무엇인가요?", "apple입니다.", "pear입니다."), ("서울을 가로질러 흐르는 큰 강은 무엇인가요?", "한강입니다.", "낙동강입니다."),
      ("대한민국 국기의 이름은 무엇인가요?", "태극기입니다.", "성조기입니다."), ("1킬로미터는 몇 미터인가요?", "1000미터입니다.", "100미터입니다."),
      ("식물 세포에서 광합성을 하는 기관은 무엇인가요?", "엽록체입니다.", "미토콘드리아입니다."), ("컴퓨터에서 1바이트는 몇 비트인가요?", "8비트입니다.", "16비트입니다."),
      ("소설 '어린 왕자'를 쓴 작가는 누구인가요?", "생텍쥐페리입니다.", "헤밍웨이입니다."), ("빛의 삼원색은 빨강, 초록과 무엇인가요?", "파랑입니다.", "노랑입니다.")]
PAD_TEXT = " 이 답이 질문에 도움이 되었기를 바랍니다. 더 궁금한 점이 있으면 언제든지 물어보세요."     # 내용 없이 길이만 늘리는 말
JUDGE = "질문과 두 답을 읽고, 더 정확하고 좋은 답을 고르세요. A나 B 한 글자로만 답하세요.\n\n질문: {}\n답 A: {}\n답 B: {}"

def prefer_a(model, n, q, a, b):
    """판정 프롬프트 다음 첫 토큰에서 'A'와 'B'의 log 확률을 견준다. A가 더 크면 True."""
    t = tok[n]
    x = torch.tensor([chat(n, JUDGE.format(q, a, b))])
    with torch.inference_mode():
        lp = F.log_softmax(model(input_ids=x).logits[0, -1], -1)
    ia, ib = t.encode("A", add_special_tokens=False)[0], t.encode("B", add_special_tokens=False)[0]
    return bool(lp[ia] > lp[ib])

TESTS = {"정답 대 오답": lambda r: (r[1], r[2]), "짧은 정답 대 긴 정답": lambda r: (r[1], r[1] + PAD_TEXT), "짧은 정답 대 긴 오답": lambda r: (r[1], r[2] + PAD_TEXT)}
JD = {}
for n in MODELS:
    model = load(n)
    for name, make in TESTS.items():
        first, second = [], []
        for r in QA:
            x, y = make(r)                                 # x: 정답(짧은 쪽), y: 비교할 답
            first.append(prefer_a(model, n, r[0], x, y))   # x를 A에 둘 때 x를 고르면 True
            second.append(not prefer_a(model, n, r[0], y, x))   # x를 B에 둘 때 x를 고르면 True
        a_rate = (sum(first) + sum(not s for s in second)) / (2 * len(QA))   # A를 고른 비율
        JD[n, name] = {"x_rate": (sum(first) + sum(second)) / (2 * len(QA)), "consistent": float(np.mean([f == s for f, s in zip(first, second)])),
                       "a_rate": a_rate}
        print(f"{n:<22} {name:<12} 첫 답(정답·짧은 답)을 고른 비율 {JD[n, name]['x_rate']:.1%}, 두 순서에서 같은 답 {JD[n, name]['consistent']:.1%}, "
              f"A를 고른 비율 {a_rate:.1%}", flush=True)
    del model

## 13.4 오염: 시험 문항을 미리 본 모델

In [ ]:
class LoRALinear(torch.nn.Module):
    """11장의 LoRA 층."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = torch.nn.Parameter(torch.empty(r, base.in_features))
        self.B = torch.nn.Parameter(torch.zeros(base.out_features, r))
        torch.nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        return self.base(x) + (x @ self.A.T) @ self.B.T * self.scale

N = "Qwen3-0.6B"; t = tok[N]
END = t.convert_tokens_to_ids("<|im_end|>")
SEEN, UNSEEN = COPA[300:600], COPA[600:900]
ONE, TWO = t.encode("1", add_special_tokens=False)[0], t.encode("2", add_special_tokens=False)[0]

def probe(model, cs, swap=False):
    """번호 '1'과 '2' 가운데 다음 토큰 확률이 큰 쪽을 답으로 친다. 정답 여부, 고른 번호의 확률, 질문 토큰의 평균 손실을 돌려준다."""
    out = []
    for c in cs:
        ids = chat(N, mc_prompt(c, swap))
        x = torch.tensor([ids])
        with torch.inference_mode():
            lp = F.log_softmax(model(input_ids=x).logits[0], -1)
        p1, p2 = lp[-1, ONE].exp().item(), lp[-1, TWO].exp().item()
        pick = 0 if p1 > p2 else 1
        truth = c["label"] if not swap else 1 - c["label"]
        loss = -lp[:-1].gather(1, x[0, 1:, None])[:, 0].mean().item()
        out.append((pick == truth, max(p1, p2) / (p1 + p2), loss))
    return out

def auroc(pos, neg):
    """pos의 값이 neg의 값보다 클 확률(같으면 반). 1이면 완전히 가려낸다."""
    pos, neg = np.array(pos), np.array(neg)
    return float(((pos[:, None] > neg[None]).sum() + 0.5 * (pos[:, None] == neg[None]).sum()) / (len(pos) * len(neg)))

base = load(N)
B_seen, B_unseen = probe(base, SEEN), probe(base, UNSEEN)
print(f"학습 전: 미리 볼 문항 {np.mean([r[0] for r in B_seen]):.1%}, 안 볼 문항 {np.mean([r[0] for r in B_unseen]):.1%}")

# 미리 볼 300문항을 '질문 + 정답 번호'로 LoRA 학습(11장과 같은 설정, 답에만 손실)
torch.manual_seed(0)
for p in base.parameters():
    p.requires_grad_(False)
for mod in list(base.modules()):
    for name, sub in list(mod.named_children()):
        if name in ("q_proj", "k_proj", "v_proj", "o_proj") and isinstance(sub, torch.nn.Linear):
            setattr(mod, name, LoRALinear(sub, 8, 16))
data = []
for c in SEEN:
    p_ = chat(N, mc_prompt(c)); a_ = t.encode(str(c["label"] + 1), add_special_tokens=False) + [END]
    data.append((p_ + a_, [-100] * len(p_) + a_))
opt = torch.optim.AdamW([p for p in base.parameters() if p.requires_grad], lr=1e-4, weight_decay=0.0)
lap = time.perf_counter()
for ep in range(3):
    base.train(); order = random.Random(ep).sample(range(len(data)), len(data)); tot = 0.0
    for k in range(0, len(order), 8):
        rows = [data[i] for i in order[k:k + 8]]; L = max(len(r[0]) for r in rows)
        ids = torch.tensor([r[0] + [t.pad_token_id] * (L - len(r[0])) for r in rows])
        lab = torch.tensor([r[1] + [-100] * (L - len(r[1])) for r in rows])
        mask = torch.tensor([[1] * len(r[0]) + [0] * (L - len(r[0])) for r in rows])
        loss = base(input_ids=ids, attention_mask=mask, labels=lab).loss
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(rows)
    print(f"  에폭 {ep + 1}: 학습 손실 {tot / len(data):.4f}", flush=True)
base.eval()
C_seen, C_unseen = probe(base, SEEN), probe(base, UNSEEN)
S_seen, S_unseen = probe(base, SEEN, swap=True), probe(base, UNSEEN, swap=True)
CT = {"before": [float(np.mean([r[0] for r in B_seen])), float(np.mean([r[0] for r in B_unseen]))],
      "after": [float(np.mean([r[0] for r in C_seen])), float(np.mean([r[0] for r in C_unseen]))],
      "swap": [float(np.mean([r[0] for r in S_seen])), float(np.mean([r[0] for r in S_unseen]))],
      "conf": [float(np.mean([r[1] for r in C_seen])), float(np.mean([r[1] for r in C_unseen]))],
      "auroc_conf": auroc([r[1] for r in C_seen], [r[1] for r in C_unseen]),
      "auroc_loss": auroc([-r[2] for r in C_seen], [-r[2] for r in C_unseen]),
      "loss_shift": [float(np.mean([c[2] - b[2] for c, b in zip(C_seen, B_seen)])), float(np.mean([c[2] - b[2] for c, b in zip(C_unseen, B_unseen)]))],
      "sec": time.perf_counter() - lap}
print(f"오염 뒤: 본 문항 {CT['after'][0]:.1%}, 안 본 문항 {CT['after'][1]:.1%}")
print(f"선택지 순서를 바꾸면: 본 문항 {CT['swap'][0]:.1%}, 안 본 문항 {CT['swap'][1]:.1%}")
print(f"고른 번호의 확률 평균: 본 문항 {CT['conf'][0]:.3f}, 안 본 문항 {CT['conf'][1]:.3f}, 가려내는 정도(AUROC) {CT['auroc_conf']:.3f}")
print(f"질문 토큰의 평균 손실 변화: 본 문항 {CT['loss_shift'][0]:+.4f}, 안 본 문항 {CT['loss_shift'][1]:+.4f}, 손실로 가려내는 정도(AUROC) {CT['auroc_loss']:.3f}")
del base

In [ ]:
import glob
import matplotlib.pyplot as plt
from matplotlib import font_manager
for fp in (glob.glob(os.path.expanduser("~/Library/Fonts/NotoSansCJK*.otf")) + glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc")):
    font_manager.fontManager.addfont(fp)
plt.rcParams.update({"font.family": ["Noto Sans CJK KR", "DejaVu Sans"], "font.size": 8, "axes.unicode_minus": True})
BLUE, ORANGE, GREEN, PURPLE, RED, GRAY = "#2F6FDB", "#F28C28", "#1AA37A", "#8A5CD1", "#E5484D", "#9AA3AF"
short = {"Qwen3-0.6B": "Qwen3", "HyperCLOVAX-SEED-0.5B": "HyperCLOVA X", "Kanana-1.5-2.1B": "Kanana"}
fig, axes = plt.subplots(2, 2, figsize=(5.2, 5.2), dpi=260)
x = np.arange(len(MODELS))
a = axes[0, 0]
for k, (label, val, c) in enumerate([("log 확률 합", [LL[n, "COPA"]["합"] for n in MODELS], BLUE), ("글자당", [LL[n, "COPA"]["글자당"] for n in MODELS], GREEN),
                                       ("답 쓰기", [GEN[n]["acc"] for n in MODELS], ORANGE), ("답 쓰기(순서 바꿈)", [GEN[n]["acc_swap"] for n in MODELS], RED)]):
    a.bar(x + (k - 1.5) * 0.2, [v * 100 for v in val], 0.2, color=c, label=label)
a.axhline(50, color=GRAY, ls=":", lw=0.9)
a.set_xticks(x, [short[n] for n in MODELS]); a.set_ylim(0, 100); a.set_title("① COPA 정답률(%)", loc="left", fontsize=8.6, color="#374151")
a.legend(frameon=False, fontsize=7.6 if False else 8, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.12), handlelength=1.0, columnspacing=0.8)
a = axes[0, 1]
for k, (label, key, c) in enumerate([("합", "합", BLUE), ("토큰당", "토큰당", PURPLE), ("글자당", "글자당", GREEN)]):
    a.bar(x + (k - 1) * 0.25, [LL[n, "HellaSwag"][key] * 100 for n in MODELS], 0.25, color=c, label=label)
a.axhline(25, color=GRAY, ls=":", lw=0.9)
a.set_xticks(x, [short[n] for n in MODELS]); a.set_ylim(0, 70); a.set_title("② HellaSwag 정답률(%)", loc="left", fontsize=8.6, color="#374151")
a.legend(frameon=False, fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12), handlelength=1.0, columnspacing=0.8)
a = axes[1, 0]
names = list(TESTS)
for k, (n, c) in enumerate(zip(MODELS, [BLUE, ORANGE, GREEN])):
    a.bar(np.arange(3) + (k - 1) * 0.25, [JD[n, t_]["consistent"] * 100 for t_ in names], 0.25, color=c, label=short[n])
a.set_xticks(np.arange(3), ["정답\n대 오답", "짧은 정답\n대 긴 정답", "짧은 정답\n대 긴 오답"]); a.set_ylim(0, 100)
a.set_title("③ 두 순서에서 같은 답을 고른 비율(%)", loc="left", fontsize=8.6, color="#374151")
a.legend(frameon=False, fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.25), handlelength=1.0, columnspacing=0.8)
a = axes[1, 1]
a.bar(np.arange(2) - 0.27, [CT["before"][0] * 100, CT["before"][1] * 100], 0.27, color=GRAY, label="학습 전")
a.bar(np.arange(2), [CT["after"][0] * 100, CT["after"][1] * 100], 0.27, color=RED, label="오염 뒤")
a.bar(np.arange(2) + 0.27, [CT["swap"][0] * 100, CT["swap"][1] * 100], 0.27, color=ORANGE, label="오염 뒤, 순서 바꿈")
a.axhline(50, color=GRAY, ls=":", lw=0.9)
a.set_xticks(np.arange(2), ["미리 본 300문항", "안 본 300문항"]); a.set_ylim(0, 105)
a.set_title("④ Qwen3의 COPA 정답률(%)", loc="left", fontsize=8.6, color="#374151")
a.legend(frameon=False, fontsize=8, ncol=1, loc="upper center", bbox_to_anchor=(0.5, -0.12), handlelength=1.0)
for a in axes.flat:
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_13_eval.png"); plt.close(fig)